# MVP v0.2.5.15: Sigmoid Reward + Guidance Scale Sweep

**Rerun of v0.2.5.2** (trajectory MSE with 10 guidance configs) but with:
1. **Sigmoid reward** (from v0.2.5.13) alongside hard binary reward
2. **Corrected scorer** (`score_timestep=5`, from v0.2.5.4)
3. **Loaded checkpoint** (skip retraining — use `mvp_v0252_traj_mse`)

v0.2.5.2 used `action_scale` 0.05–0.5 but with the broken `score_timestep=1`.
v0.2.5.6/13 used the corrected scorer but only `action_scale=0.001`.
This experiment bridges that gap.

In [ ]:
%matplotlib inline
import sys, os, importlib, time, math, json
import numpy as np
import torch
import torch.nn as nn
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# ── Configuration ──
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]
STATE_DIM = 19
ACTION_DIM = 7
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 26

# Paths
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
CKPT_DIR = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/test/20260309132349"
TARGET_ROLLOUT_DIR = PROJECT_ROOT / "rollouts" / "target_policy_50"
DIFFUSION_CKPT_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v0252_traj_mse"
ORACLE_JSON = CKPT_DIR / "oracle_50.json"

NUM_TARGET_ROLLOUTS = 50
CHUNK_SIZE = 4
FRAME_STACK = 1
STRIDE = 2

# Diffusion architecture (must match saved checkpoint)
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False

# Generation
NUM_SYNTHETIC_TRAJS = 50
T_GEN = 60
GAMMA = 1.0

# Reward
CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84
SIGMOID_SHARPNESS = 150.0

# Guidance — CORRECTED: score_timestep=5
SCORE_TIMESTEP = 5
K_GUIDE = 1
NORMALIZE_GRAD = True
USE_ADAPTIVE = False
CLAMP = False
L_INF = 1.0

# Same 10 configs as v0.2.5.2
GUIDANCE_CONFIGS = [
    {"action_scale": 0.0,  "ratio": 0.0,  "label": "unguided"},
    {"action_scale": 0.05, "ratio": 0.0,  "label": "pos_only_0.05"},
    {"action_scale": 0.1,  "ratio": 0.0,  "label": "pos_only_0.1"},
    {"action_scale": 0.2,  "ratio": 0.0,  "label": "pos_only_0.2"},
    {"action_scale": 0.05, "ratio": 0.25, "label": "full_0.05_r0.25"},
    {"action_scale": 0.1,  "ratio": 0.25, "label": "full_0.1_r0.25"},
    {"action_scale": 0.2,  "ratio": 0.25, "label": "full_0.2_r0.25"},
    {"action_scale": 0.2,  "ratio": 0.5,  "label": "full_0.2_r0.5"},
    {"action_scale": 0.5,  "ratio": 0.25, "label": "full_0.5_r0.25"},
    {"action_scale": 0.5,  "ratio": 0.5,  "label": "full_0.5_r0.5"},
]

DIM_NAMES = {
    0: "cube_x", 1: "cube_y", 2: "cube_z",
    3: "cube_qx", 4: "cube_qy", 5: "cube_qz", 6: "cube_qw",
    7: "grip2cube_x", 8: "grip2cube_y", 9: "grip2cube_z",
    10: "eef_x", 11: "eef_y", 12: "eef_z",
    13: "eef_qx", 14: "eef_qy", 15: "eef_qz", 16: "eef_qw",
    17: "gripper_q0", 18: "gripper_q1",
}

print(f"Config: score_timestep={SCORE_TIMESTEP}, sigmoid_sharpness={SIGMOID_SHARPNESS}")
print(f"Guidance configs to sweep: {len(GUIDANCE_CONFIGS)}")

In [ ]:
# ── Reward functions ──

def hard_reward(cube_z):
    """Binary reward: 1 if cube_z > 0.84, else 0."""
    return (cube_z > LIFT_THRESHOLD).astype(np.float32)

def sigmoid_reward(cube_z, center=LIFT_THRESHOLD, sharpness=SIGMOID_SHARPNESS):
    """Smooth sigmoid reward centered at the lift threshold."""
    return (1.0 / (1.0 + np.exp(-sharpness * (cube_z - center)))).astype(np.float32)

def compute_sr_hard(states):
    """Fraction of trajectories where cube_z ever exceeds threshold."""
    B = states.shape[0]
    return np.mean([np.any(states[j, :, CUBE_Z_INDEX] > LIFT_THRESHOLD) for j in range(B)])

def compute_ope_hard(states, gamma=1.0):
    """Per-trajectory discounted return using hard binary reward."""
    B, T, _ = states.shape
    cube_z = states[:, :, CUBE_Z_INDEX]
    rewards = hard_reward(cube_z)
    gammas = gamma ** np.arange(T)
    return (rewards * gammas[None, :]).sum(axis=1)

def compute_ope_sigmoid(states, gamma=1.0):
    """Per-trajectory discounted return using sigmoid reward."""
    B, T, _ = states.shape
    cube_z = states[:, :, CUBE_Z_INDEX]
    rewards = sigmoid_reward(cube_z)
    gammas = gamma ** np.arange(T)
    return (rewards * gammas[None, :]).sum(axis=1)

# Verify sigmoid behavior
print("Sigmoid reward at key cube_z values:")
for z in [0.82, 0.835, 0.838, 0.84, 0.842, 0.85]:
    print(f"  cube_z={z:.3f}  hard={hard_reward(np.array([z]))[0]:.0f}  sigmoid={sigmoid_reward(np.array([z]))[0]:.4f}")

In [ ]:
# ── Load oracle + target rollouts + expert demos ──

with open(ORACLE_JSON, "r") as f:
    oracle_data = json.load(f)
oracle_returns = np.array(oracle_data["returns"])
oracle_value = float(oracle_data["mean_return"])
oracle_success_rate = float(np.mean(oracle_returns > 0.5))
print(f"Oracle V^pi = {oracle_value:.4f}, SR = {oracle_success_rate*100:.1f}%")

# Load checkpoint
ckpt = load_checkpoint(CKPT_DIR, ckpt_path=Path("last.pth"))

# Part A: Target rollouts
target_data = []
all_states_list = []
all_actions_list = []

rollout_paths = sorted(TARGET_ROLLOUT_DIR.glob("rollout_*.h5"))[:NUM_TARGET_ROLLOUTS]
assert len(rollout_paths) >= NUM_TARGET_ROLLOUTS, \
    f"Expected {NUM_TARGET_ROLLOUTS} rollouts, found {len(rollout_paths)}"

for path in tqdm(rollout_paths, desc="Loading target rollouts"):
    with h5py.File(path, "r") as f:
        latents = f["latents"][:]
        actions = f["actions"][:]
        rewards = f["rewards"][:] if "rewards" in f else np.zeros(len(actions))
    states = latents[:, -1, :] if latents.ndim == 3 else latents
    states = states.astype(np.float32)
    actions = actions.astype(np.float32)
    episode = {
        "states": states[:-1], "actions": actions[:-1],
        "rewards": rewards[:-1] if len(rewards) > 1 else rewards,
        "next-states": states[1:],
    }
    target_data.append(episode)
    all_states_list.append(states)
    all_actions_list.append(actions)

print(f"Loaded {len(target_data)} target episodes")

# Part B: Expert demos
expert_data = []
with h5py.File(DEMO_HDF5, "r") as f:
    demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
    for dk in tqdm(demo_keys, desc="Loading expert demos"):
        demo = f[f"data/{dk}"]
        obs_arrays = [demo["obs"][k][:].astype(np.float32) for k in OBS_KEYS]
        states = np.concatenate(obs_arrays, axis=-1)
        actions = demo["actions"][:].astype(np.float32)
        rewards = demo["rewards"][:].astype(np.float32)
        episode = {
            "states": states[:-1], "actions": actions[:-1],
            "rewards": rewards[:-1], "next-states": states[1:],
        }
        expert_data.append(episode)
        all_states_list.append(states)
        all_actions_list.append(actions)

print(f"Loaded {len(expert_data)} expert episodes")

# Normalization
all_episodes = target_data + expert_data
all_states = np.concatenate(all_states_list, axis=0)
all_actions = np.concatenate(all_actions_list, axis=0)
total_transitions = sum(len(ep["states"]) for ep in all_episodes)

norm_mean = np.concatenate([all_states.mean(0), all_actions.mean(0)]).astype(np.float32)
norm_std = np.maximum(np.concatenate([all_states.std(0), all_actions.std(0)]), 1e-6).astype(np.float32)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t

target_sr = np.mean([np.any(ep["states"][:, CUBE_Z_INDEX] > LIFT_THRESHOLD) for ep in target_data])
expert_sr = np.mean([np.any(ep["states"][:, CUBE_Z_INDEX] > LIFT_THRESHOLD) for ep in expert_data])

print(f"\nCombined: {len(all_episodes)} episodes, {total_transitions} transitions")
print(f"Target SR (in recorded data): {target_sr*100:.1f}%, Expert SR: {expert_sr*100:.1f}%")

In [ ]:
# ── Load pre-trained diffuser checkpoint ──
# (Skip training — reuse v0.2.5.2's saved model)

temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE, transition_dim=TRANSITION_DIM,
    dim=BASE_DIM, dim_mults=DIM_MULTS, attention=False,
).to(device)

diffusion_model = GaussianDiffusion(
    model=temporal_model, horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM, action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS, normalizer=normalize_fn,
    unnormalizer=unnormalize_fn, predict_epsilon=PREDICT_EPSILON,
    loss_type="l2", clip_denoised=False,
    action_weight=ACTION_WEIGHT, loss_weights=None, loss_discount=1.0,
).to(device)

# Load EMA checkpoint
ema_path = DIFFUSION_CKPT_DIR / "diffusion_model_ema.pt"
assert ema_path.exists(), f"EMA checkpoint not found: {ema_path}"
ema_state = torch.load(ema_path, map_location=device)
diffusion_model.load_state_dict(ema_state)
diffusion_model.eval()

n_params = sum(p.numel() for p in diffusion_model.parameters())
print(f"Loaded EMA diffuser from {ema_path}")
print(f"  {n_params:,} parameters")

In [ ]:
# ── BC behavior policy + target scorer ──

class BCGaussian(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)

    def forward(self, state):
        h = self.net(state)
        return self.mean_head(h), self.log_std_head(h).clamp(-5, 2)

    def log_prob(self, state, action):
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        return -0.5 * (((action - mean) / std) ** 2 + 2 * log_std + math.log(2 * math.pi)).sum(dim=-1)

    def grad_log_prob(self, state, action):
        with torch.no_grad():
            mean, log_std = self.forward(state)
            std = torch.exp(log_std)
            return -(action - mean) / (std ** 2)

    def grad_log_prob_chunk(self, states, actions):
        B, T, _ = states.shape
        grad_flat = self.grad_log_prob(states.reshape(B*T, -1), actions.reshape(B*T, -1))
        return grad_flat.reshape(B, T, -1)

# Train BC on target rollout data
demo_states = np.concatenate([ep["states"] for ep in target_data], axis=0)
demo_actions = np.concatenate([ep["actions"] for ep in target_data], axis=0)
print(f"BC training data: {demo_states.shape[0]} (state, action) pairs")

bc_behavior = BCGaussian(STATE_DIM, ACTION_DIM, hidden_dim=256).to(device)
bc_optimizer = torch.optim.Adam(bc_behavior.parameters(), lr=1e-3)
states_t = torch.tensor(demo_states, dtype=torch.float32, device=device)
actions_t = torch.tensor(demo_actions, dtype=torch.float32, device=device)

BC_EPOCHS = 500
BC_BATCH = 256
bc_losses = []
print(f"Training BC_Gaussian for {BC_EPOCHS} epochs...")
bc_behavior.train()
for epoch in range(BC_EPOCHS):
    idx = torch.randint(0, len(states_t), (BC_BATCH,), device=device)
    nll = -bc_behavior.log_prob(states_t[idx], actions_t[idx]).mean()
    bc_optimizer.zero_grad()
    nll.backward()
    bc_optimizer.step()
    bc_losses.append(nll.item())
bc_behavior.eval()
print(f"Final NLL: {bc_losses[-1]:.4f}")

# Target scorer with CORRECTED score_timestep=5
target_algo = build_algo_from_checkpoint(ckpt, device=str(device))
target_scorer = RobomimicDiffusionScorer(
    target_algo, device=str(device), obs_keys=OBS_KEYS,
    score_timestep=SCORE_TIMESTEP,
)
print(f"Target scorer: score_timestep={SCORE_TIMESTEP}, sigma={target_scorer.sigma:.6f}")

In [ ]:
# ── Generate trajectories with guidance sweep ──

def generate_trajectories_full_guidance(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim,
    chunk_size, t_gen, device,
    target_scorer=None, behavior_scorer=None,
    action_scale=0.0, ratio=0.5,
    k_guide=1, normalize_grad=True,
    use_adaptive=False, clamp=False, l_inf=1.0,
):
    guided = (target_scorer is not None and action_scale > 0)
    use_neg_grad = (behavior_scorer is not None and ratio > 0)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    n_timesteps = diffusion_model.n_timesteps

    padded = torch.cat([initial_states, torch.zeros(batch_size, action_dim, device=device)], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]

    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0

    while total_generated < t_gen:
        shape = (batch_size, chunk_size, transition_dim)
        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_diff in reversed(range(n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)
            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)

            if guided:
                model_mean = unnormalize_fn(model_mean)
                for _k in range(k_guide):
                    states_chunk = model_mean[:, :, :state_dim]
                    actions_chunk = model_mean[:, :, state_dim:]
                    target_grad = target_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                    if use_neg_grad:
                        behavior_grad = behavior_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                    if normalize_grad:
                        eps = 1e-6
                        target_grad = target_grad / (target_grad.norm(dim=-1, keepdim=True) + eps)
                        if use_neg_grad:
                            behavior_grad = behavior_grad / (behavior_grad.norm(dim=-1, keepdim=True) + eps)
                    if clamp:
                        target_grad = target_grad.clamp(-l_inf, l_inf)
                        if use_neg_grad:
                            behavior_grad = behavior_grad.clamp(-l_inf, l_inf)
                    guide_actions = target_grad - ratio * behavior_grad if use_neg_grad else target_grad
                    guide = torch.zeros_like(model_mean)
                    guide[:, :, state_dim:] = guide_actions
                    model_mean = model_mean + action_scale * guide
                    model_mean = normalize_fn(model_mean)
                    model_mean = apply_conditioning(model_mean, conditions, state_dim)
                    model_mean = unnormalize_fn(model_mean)
                model_mean = normalize_fn(model_mean)

            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)

        chunk_unnorm = unnormalize_fn(x)
        steps_remaining = t_gen - total_generated
        steps_to_store = min(chunk_size - 1, steps_remaining)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store

        if total_generated >= t_gen:
            break
        last_states_norm = x[:, -1, :state_dim]
        conditions = {0: last_states_norm}

    return all_trajectories.detach().cpu().numpy()


# Build paired real trajectory array
N_COMPARE = min(NUM_SYNTHETIC_TRAJS, len(target_data))
real_trajs = []
for ep in target_data[:N_COMPARE]:
    traj = np.concatenate([ep["states"], ep["actions"]], axis=-1)
    T = len(traj)
    if T >= T_GEN:
        real_trajs.append(traj[:T_GEN])
    else:
        padded = np.zeros((T_GEN, TRANSITION_DIM), dtype=np.float32)
        padded[:T] = traj
        padded[T:] = traj[-1]
        real_trajs.append(padded)
real_trajs = np.array(real_trajs)
real_states = real_trajs[:, :, :STATE_DIM]
real_actions = real_trajs[:, :, STATE_DIM:]

# Initial states from real target rollouts
np.random.seed(42)
torch.manual_seed(42)
real_initial_states = np.array([ep["states"][0] for ep in target_data[:N_COMPARE]])
initial_states_t = torch.tensor(real_initial_states, dtype=torch.float32, device=device)
print(f"Paired comparison: {N_COMPARE} trajectories, {T_GEN} steps each")

# ── Run guidance sweep ──
results_by_config = {}

for i, gc in enumerate(GUIDANCE_CONFIGS):
    label = gc["label"]
    action_scale = gc["action_scale"]
    ratio = gc["ratio"]
    print(f"\n[{i+1}/{len(GUIDANCE_CONFIGS)}] {label}")

    t0 = time.time()
    trajs = generate_trajectories_full_guidance(
        diffusion_model=diffusion_model,
        initial_states=initial_states_t,
        normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
        state_dim=STATE_DIM, action_dim=ACTION_DIM,
        chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
        target_scorer=target_scorer if action_scale > 0 else None,
        behavior_scorer=bc_behavior if (action_scale > 0 and ratio > 0) else None,
        action_scale=action_scale, ratio=ratio,
        k_guide=K_GUIDE, normalize_grad=NORMALIZE_GRAD,
        use_adaptive=USE_ADAPTIVE, clamp=CLAMP, l_inf=L_INF,
    )
    elapsed = time.time() - t0

    synth_states = trajs[:, :, :STATE_DIM]
    synth_actions = trajs[:, :, STATE_DIM:]

    # MSE metrics
    init_diff = np.abs(real_states[:, 0, :] - synth_states[:, 0, :]).max()
    per_step_mse = np.mean((real_states - synth_states) ** 2, axis=(0, 2))
    per_dim_mse = np.mean((real_states - synth_states) ** 2, axis=(0, 1))
    per_traj_mse = np.mean((real_states - synth_states) ** 2, axis=(1, 2))
    total_state_mse = np.mean((real_states - synth_states) ** 2)

    # Hard + sigmoid reward
    sr_hard = compute_sr_hard(synth_states)
    ope_hard = compute_ope_hard(synth_states, GAMMA)
    ope_sigmoid = compute_ope_sigmoid(synth_states, GAMMA)

    results_by_config[label] = {
        "trajs": trajs,
        "synth_states": synth_states,
        "synth_actions": synth_actions,
        "per_step_mse": per_step_mse,
        "per_dim_mse": per_dim_mse,
        "per_traj_mse": per_traj_mse,
        "state_mse": total_state_mse,
        "state_rmse": np.sqrt(total_state_mse),
        "sr_hard": sr_hard,
        "ope_hard_mean": float(ope_hard.mean()),
        "ope_hard_std": float(ope_hard.std()),
        "ope_sigmoid_mean": float(ope_sigmoid.mean()),
        "ope_sigmoid_std": float(ope_sigmoid.std()),
        "init_diff": init_diff,
        "time": elapsed,
        "action_scale": action_scale,
        "ratio": ratio,
    }

    print(f"  MSE={total_state_mse:.6f}, SR={sr_hard*100:.0f}%, "
          f"hard_OPE={ope_hard.mean():.3f}, sigmoid_OPE={ope_sigmoid.mean():.3f}, "
          f"time={elapsed:.0f}s")

# ── Summary table ──
print(f"\n{'='*100}")
print("GUIDANCE SWEEP — TRAJECTORY MSE + SIGMOID REWARD SUMMARY")
print(f"{'='*100}")
print(f"Oracle V^pi = {oracle_value:.4f} (SR={oracle_success_rate*100:.0f}%), score_timestep={SCORE_TIMESTEP}")
print(f"\n{'Config':<22} {'Scale':>6} {'Ratio':>6} {'MSE':>10} {'SR':>6} {'Hard OPE':>10} {'Sig OPE':>10} {'Hard RelErr':>12} {'Sig RelErr':>12}")
print("-" * 100)
for label, r in results_by_config.items():
    hard_rel = abs(r['sr_hard'] - oracle_value) / oracle_value * 100 if oracle_value > 0 else 0
    # For sigmoid: compare mean sigmoid OPE. Oracle sigmoid OPE is not directly available,
    # so we report the raw values and compare SR-based relative error.
    print(f"{label:<22} {r['action_scale']:>6.2f} {r['ratio']:>6.2f} "
          f"{r['state_mse']:>10.6f} {r['sr_hard']*100:>5.0f}% "
          f"{r['ope_hard_mean']:>10.3f} {r['ope_sigmoid_mean']:>10.3f} "
          f"{hard_rel:>11.1f}% {r['ope_sigmoid_std']:>11.3f}")

In [ ]:
# ── Figure 1: MSE + SR + OPE bar charts ──

labels = list(results_by_config.keys())
mses = [results_by_config[l]["state_mse"] for l in labels]
srs = [results_by_config[l]["sr_hard"] * 100 for l in labels]
ope_hards = [results_by_config[l]["ope_hard_mean"] for l in labels]
ope_sigs = [results_by_config[l]["ope_sigmoid_mean"] for l in labels]

colors = ["steelblue" if "unguided" in l else ("orange" if "pos_only" in l else "coral") for l in labels]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# State MSE
axes[0, 0].bar(range(len(labels)), mses, color=colors, edgecolor="black")
axes[0, 0].set_xticks(range(len(labels)))
axes[0, 0].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[0, 0].set_ylabel("State MSE")
axes[0, 0].set_title("State MSE by Guidance Config")
axes[0, 0].grid(True, alpha=0.3, axis="y")

# Hard SR
axes[0, 1].bar(range(len(labels)), srs, color=colors, edgecolor="black")
axes[0, 1].axhline(y=oracle_value*100, color="green", linestyle="--", lw=2, label=f"Oracle={oracle_value*100:.0f}%")
axes[0, 1].set_xticks(range(len(labels)))
axes[0, 1].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[0, 1].set_ylabel("Success Rate (%)")
axes[0, 1].set_title("Hard SR by Guidance Config")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3, axis="y")

# Hard OPE
axes[1, 0].bar(range(len(labels)), ope_hards, color=colors, edgecolor="black")
axes[1, 0].set_xticks(range(len(labels)))
axes[1, 0].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[1, 0].set_ylabel("Mean Return")
axes[1, 0].set_title("Hard OPE (sum of per-step binary reward)")
axes[1, 0].grid(True, alpha=0.3, axis="y")

# Sigmoid OPE
axes[1, 1].bar(range(len(labels)), ope_sigs, color=colors, edgecolor="black")
axes[1, 1].set_xticks(range(len(labels)))
axes[1, 1].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[1, 1].set_ylabel("Mean Return")
axes[1, 1].set_title(f"Sigmoid OPE (sharpness={SIGMOID_SHARPNESS})")
axes[1, 1].grid(True, alpha=0.3, axis="y")

plt.suptitle(f"v0.2.5.15: Guidance Sweep with Corrected Scorer (t={SCORE_TIMESTEP}) + Sigmoid Reward",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Per-step MSE curves ──

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for label, r in results_by_config.items():
    color = "steelblue" if "unguided" in label else ("orange" if "pos_only" in label else "coral")
    alpha = 1.0 if "unguided" in label else 0.6
    lw = 2.5 if "unguided" in label else 1.5
    axes[0].plot(r["per_step_mse"], label=label, color=color, alpha=alpha, linewidth=lw)
    axes[1].plot(np.sqrt(r["per_step_mse"]), label=label, color=color, alpha=alpha, linewidth=lw)

axes[0].set_xlabel("Timestep"); axes[0].set_ylabel("MSE")
axes[0].set_title("Per-Step State MSE"); axes[0].legend(fontsize=6); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("Timestep"); axes[1].set_ylabel("RMSE")
axes[1].set_title("Per-Step State RMSE"); axes[1].legend(fontsize=6); axes[1].grid(True, alpha=0.3)

plt.suptitle("v0.2.5.15: Error Accumulation Over Time", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 3: Per-dimension MSE (unguided vs best MSE) ──

best_mse_label = min(results_by_config, key=lambda k: results_by_config[k]["state_mse"])
best_r = results_by_config[best_mse_label]
unguided_r = results_by_config["unguided"]
dim_labels = [DIM_NAMES[d] for d in range(STATE_DIM)]

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
x_pos = np.arange(STATE_DIM)
width = 0.35
ax.bar(x_pos - width/2, unguided_r["per_dim_mse"], width, color="steelblue",
       edgecolor="black", alpha=0.8, label="Unguided")
ax.bar(x_pos + width/2, best_r["per_dim_mse"], width, color="coral",
       edgecolor="black", alpha=0.8, label=f"Best MSE ({best_mse_label})")
ax.set_xticks(x_pos)
ax.set_xticklabels(dim_labels, rotation=60, ha="right", fontsize=8)
ax.set_ylabel("MSE")
ax.set_title(f"Per-Dimension State MSE: Unguided vs {best_mse_label}")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"Best MSE config: {best_mse_label} (MSE={best_r['state_mse']:.6f})")
print(f"Unguided MSE: {unguided_r['state_mse']:.6f}")

In [ ]:
# ── Figure 4: Trajectory overlays (key dims) ──

KEY_DIMS = {
    "cube_z": 2, "cube_x": 0, "eef_z": 12, "eef_x": 10,
    "grip2cube_z": 9, "gripper_q0": 17,
}
N_SHOW = 10
unguided_states = unguided_r["synth_states"]
best_states = best_r["synth_states"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for idx, (name, dim) in enumerate(KEY_DIMS.items()):
    ax = axes[idx]
    for j in range(min(N_SHOW, N_COMPARE)):
        ax.plot(real_states[j, :, dim], color="green", alpha=0.15,
                label="Real" if j == 0 else "")
        ax.plot(unguided_states[j, :, dim], color="steelblue", alpha=0.15,
                label="Unguided" if j == 0 else "")
        ax.plot(best_states[j, :, dim], color="coral", alpha=0.15,
                label=f"Best ({best_mse_label})" if j == 0 else "")
    ax.plot(real_states[:, :, dim].mean(axis=0), color="darkgreen", linewidth=2.5)
    ax.plot(unguided_states[:, :, dim].mean(axis=0), color="darkblue", linewidth=2.5, linestyle="--")
    ax.plot(best_states[:, :, dim].mean(axis=0), color="darkred", linewidth=2.5, linestyle=":")
    if dim == CUBE_Z_INDEX:
        ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Timestep"); ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7, loc="upper left")

plt.suptitle("v0.2.5.15: Real (green) vs Unguided (blue) vs Best MSE (red)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 5: Cube z across ALL configs ──

n_configs = len(results_by_config)
n_cols = 4
n_rows = math.ceil(n_configs / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for idx, (label, r) in enumerate(results_by_config.items()):
    ax = axes[idx]
    synth_s = r["synth_states"]
    for j in range(min(5, N_COMPARE)):
        ax.plot(real_states[j, :, CUBE_Z_INDEX], color="green", alpha=0.1)
    for j in range(min(15, N_COMPARE)):
        ax.plot(synth_s[j, :, CUBE_Z_INDEX], color="coral", alpha=0.2)
    ax.plot(synth_s[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkred", linewidth=2, label="Synth mean")
    ax.plot(real_states[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkgreen", linewidth=2,
            linestyle="--", label="Real mean")
    ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(f"{label}\nMSE={r['state_mse']:.5f}, SR={r['sr_hard']*100:.0f}%\n"
                 f"sig_OPE={r['ope_sigmoid_mean']:.2f}", fontsize=8)
    ax.set_xlabel("Timestep"); ax.set_ylabel("cube_z")
    ax.set_ylim([0.78, 0.95]); ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7)

for idx in range(n_configs, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("v0.2.5.15: Cube Z Across All Guidance Configs", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 6: Hard vs Sigmoid OPE comparison ──

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: hard OPE vs sigmoid OPE
for label, r in results_by_config.items():
    color = "steelblue" if "unguided" in label else ("orange" if "pos_only" in label else "coral")
    axes[0].scatter(r["ope_hard_mean"], r["ope_sigmoid_mean"], color=color, s=80,
                    edgecolor="black", zorder=5)
    axes[0].annotate(label, (r["ope_hard_mean"], r["ope_sigmoid_mean"]),
                     fontsize=6, ha="left", va="bottom")
axes[0].set_xlabel("Hard OPE (mean return)")
axes[0].set_ylabel("Sigmoid OPE (mean return)")
axes[0].set_title("Hard vs Sigmoid OPE")
axes[0].grid(True, alpha=0.3)

# Bar: sigmoid OPE spread
sig_means = [results_by_config[l]["ope_sigmoid_mean"] for l in labels]
sig_stds = [results_by_config[l]["ope_sigmoid_std"] for l in labels]
axes[1].bar(range(len(labels)), sig_means, yerr=sig_stds, color=colors,
            edgecolor="black", capsize=3)
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[1].set_ylabel("Sigmoid OPE")
axes[1].set_title(f"Sigmoid OPE with Std Bars (sharpness={SIGMOID_SHARPNESS})")
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("v0.2.5.15: Hard vs Sigmoid Reward Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 7: Sigmoid OPE distribution per config (histograms) ──

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for idx, (label, r) in enumerate(results_by_config.items()):
    ax = axes[idx]
    synth_s = r["synth_states"]
    sig_returns = compute_ope_sigmoid(synth_s, GAMMA)
    hard_returns = compute_ope_hard(synth_s, GAMMA)
    ax.hist(sig_returns, bins=20, alpha=0.6, color="coral", label="Sigmoid")
    ax.hist(hard_returns, bins=20, alpha=0.4, color="steelblue", label="Hard")
    ax.set_title(f"{label}\nsig={sig_returns.mean():.2f}, hard={hard_returns.mean():.2f}", fontsize=8)
    ax.set_xlabel("Return")
    if idx == 0:
        ax.legend(fontsize=7)

plt.suptitle("v0.2.5.15: Return Distributions (Hard vs Sigmoid)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Final summary ──

# Marginal statistics for unguided
print(f"{'Dim':<15} {'Real Mean':>10} {'Synth Mean':>11} {'Real Std':>10} {'Synth Std':>11} {'MSE':>10}")
print("-" * 70)
for d in range(STATE_DIM):
    r_mean = real_states[:, :, d].mean()
    s_mean = unguided_r["synth_states"][:, :, d].mean()
    r_std = real_states[:, :, d].std()
    s_std = unguided_r["synth_states"][:, :, d].std()
    mse = unguided_r["per_dim_mse"][d]
    print(f"{DIM_NAMES[d]:<15} {r_mean:>10.5f} {s_mean:>11.5f} {r_std:>10.5f} {s_std:>11.5f} {mse:>10.6f}")

print(f"\n{'='*80}")
print(f"MVP v0.2.5.15 COMPLETE")
print(f"{'='*80}")
print(f"Oracle V^pi = {oracle_value:.4f} (SR={oracle_success_rate*100:.0f}%)")
print(f"Score timestep = {SCORE_TIMESTEP} (corrected from v0.2.5.2's default=1)")
print(f"Sigmoid sharpness = {SIGMOID_SHARPNESS}")
print(f"\nKey comparison vs v0.2.5.2 (which used score_timestep=1):")
print(f"  v0.2.5.2 best MSE: pos_only_0.1 (MSE=0.005044, SR=24%)")
print(f"  v0.2.5.15 best MSE: {best_mse_label} (MSE={best_r['state_mse']:.6f}, SR={best_r['sr_hard']*100:.0f}%)")
print(f"\n  v0.2.5.2 unguided: MSE=0.006732, SR=60%")
print(f"  v0.2.5.15 unguided: MSE={unguided_r['state_mse']:.6f}, SR={unguided_r['sr_hard']*100:.0f}%")
print(f"\nSigmoid OPE spread across configs:")
sig_vals = [r['ope_sigmoid_mean'] for r in results_by_config.values()]
print(f"  min={min(sig_vals):.3f}, max={max(sig_vals):.3f}, range={max(sig_vals)-min(sig_vals):.3f}")